In [1]:
# Importing the required libraries

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import cdist
from scipy.spatial.distance import jaccard
from time import time

In [2]:
# Loading Data

data = pd.read_csv("data.csv")
labels = pd.read_csv("label.csv")

In [3]:
data.shape, labels.shape

((9999, 784), (9999, 1))

In [4]:
print(labels.head())

   7
0  2
1  1
2  0
3  4
4  1


In [5]:
# Checking the number of unique labels

k = labels['7'].nunique()
print("The number of centroids to be initialized is: ", k)

The number of centroids to be initialized is:  10


In [6]:
# Dropping the columns that have the same minimum and maximum values because they are not useful

cols = data.columns
for c in cols:
    if data[c].min() == data[c].max():
        data.drop(columns = c, inplace = True)

In [7]:
# Defining the distance metrics and centroid update rule

def euclideanDist(p1, p2):
    return np.sqrt(np.sum((p1 - p2) ** 2))

def cosineDist(p1, p2):
    return 1 - (np.dot(p1, p2) / np.sqrt(np.dot(p1, p1) * np.dot(p2, p2)))

def jaccardDist(p1, p2):
    return 1 - (np.sum(np.minimum(p1, p2)) / np.sum(np.maximum(p1, p2)))

def centroidUpdate(points):
    return np.average(points, axis = 0)

In [8]:
# K-means Implementation

def k_means(data, k, initial_centroids, distance, max_iterations=100, tol=1e-4):
    """
    Improved K-Means clustering algorithm.
    
    Parameters:
    data (array-like): Input data points.
    k (int): Number of clusters.
    initial_centroids (array-like): Initial centroids for each cluster.
    distance (function): Distance function between data points and centroids.
    max_iterations (int): Maximum number of iterations (prevents infinite loops).
    tol (float): Tolerance for centroid convergence.
    
    Returns:
    centroids (numpy.ndarray): Final centroids.
    cluster_indices (numpy.ndarray): Index of the cluster each data point belongs to.
    """
    # Convert inputs to numpy arrays for consistent handling
    data = np.asarray(data)
    centroids = np.asarray(initial_centroids)
    
    # Validate inputs
    if len(centroids) != k:
        raise ValueError("Number of initial centroids must equal k.")
    if k <= 0:
        raise ValueError("k must be a positive integer.")
    
    prev_clusters = None
    iterations = 0
    
    while iterations < max_iterations:
        # Calculate distances from each point to centroids
        distances = np.array([[distance(x, c) for c in centroids] for x in data])
        # Assign each point to the nearest cluster
        cluster_indices = np.argmin(distances, axis=1)
        
        # Update centroids
        new_centroids = []
        for i in range(k):
            cluster_points = data[cluster_indices == i]
            if len(cluster_points) == 0:
                # Handle empty clusters by retaining the current centroid
                new_centroids.append(centroids[i])
            else:
                new_centroids.append(cluster_points.mean(axis=0))
        new_centroids = np.array(new_centroids)
        
        # Check for convergence (using tolerance)
        if np.allclose(centroids, new_centroids, atol=tol):
            break
        centroids = new_centroids.copy()
        iterations += 1
    
    return centroids, cluster_indices

In [9]:
# Converting dataframe to array 

dataArray = data.to_numpy()
labelsArray = [labels.to_numpy()[i][0] for i in range(labels.shape[0])]

# Initialize k random points as initial centroids

centroidsInitial = []
for _ in range(k):
    i = np.random.randint(dataArray.shape[0])
    centroidsInitial.append(dataArray[i])

In [10]:
# Performing K-means with the different distance metrics

centroidsEuclidean, clustersEuclidean = k_means(dataArray, k, centroidsInitial, euclideanDist)
centroidsCosine, clustersCosine = k_means(dataArray, k, centroidsInitial, cosineDist)
centroidsJaccard, clustersJaccard = k_means(dataArray, k, centroidsInitial, jaccardDist)

**Q1: Run K-means clustering with Euclidean, Cosine and Jarcard similarity. Specify K= the number of categorical values of y (the number of classifications). Compare the SSEs of Euclidean-K-means, Cosine-K-means, Jarcard-K-means. Which method is better?**

In [11]:
# Calculating SSE for all three distance metrics

def SSE(points, centroids, clustersIndex):
    sse = 0
    for i in range(len(points)):
        sse += np.sum((centroids[clustersIndex[i]] - points[i]) ** 2)

    return sse

sse_df = pd.DataFrame({
    "Distance_Metric": ["Euclidean", "Cosine", "Jaccard"],
    "SSE": [
        SSE(dataArray, centroidsEuclidean, clustersEuclidean),
        SSE(dataArray, centroidsCosine, clustersCosine),
        SSE(dataArray, centroidsJaccard, clustersJaccard)
    ]
})
sse_df = sse_df.sort_values(by = ["SSE"])
sse_df

,Distance_Metric,SSE
2,Jaccard,2.541436e+10
0,Euclidean,2.546966e+10
1,Cosine,2.565005e+10


**Q2: Compare the accuracies of Euclidean-K-means Cosine-K-means, Jarcard-K-means. First, label each cluster using the majority vote label of the data points in that cluster. Later, compute the predictive accuracy of Euclidean-K-means, Cosine-K-means, Jarcard-K-means. Which metric is better?**

In [12]:
def predictLabels(labels, k, clustersIndex):
    cluster_indexLabel = [[0 for _ in range(k)] for _ in range(k)]

    for i in range(len(clustersIndex)):
        cluster_indexLabel[clustersIndex[i]][labels[i]] += 1

    predicted_clusterLabels = np.argmax(cluster_indexLabel, axis = 1)

    predictedLabels = []
    for i in range(len(clustersIndex)):
        predictedLabels.append(predicted_clusterLabels[clustersIndex[i]])

    return predictedLabels

In [13]:
def accuracy(predictedLabels, groundTruth):
    return np.average(predictedLabels == np.array(groundTruth)) * 100

In [14]:
# Predicting the labels for all 3 distance metrics

labelsEuclidean = predictLabels(labelsArray, k, clustersEuclidean)
labelsCosine = predictLabels(labelsArray, k, clustersCosine)
labelsJaccard = predictLabels(labelsArray, k, clustersJaccard)

In [15]:
# Calculating accuracies for all three distance metrics

accuracy_df = pd.DataFrame({
    "Distance_Metric": ["Euclidean", "Cosine", "Jaccard"],
    "Accuracy": [
        accuracy(labelsEuclidean, labelsArray),
        accuracy(labelsCosine, labelsArray),
        accuracy(labelsJaccard, labelsArray)
    ]
})
accuracy_df = accuracy_df.sort_values(by = ["Accuracy"], ascending = False)
accuracy_df

,Distance_Metric,Accuracy
2,Jaccard,60.236024
1,Cosine,60.106011
0,Euclidean,54.065407


**Q3: Set up the same stop criteria: “when there is no change in centroid position OR when the SSE value increases in the next iteration OR when the maximum preset value (e.g., 500, you can set the preset value by yourself) of iteration is complete”, for Euclidean-K-means, Cosine-K-means, Jarcard-K-means. Which method requires more iterations and times to converge?**

In [16]:
import numpy as np
from time import time

def kMeans_nochange(data, k, centroidsInitial, distance, max_iterations=100, tol=1e-4):
    start_time = time()
    
    # Convert inputs to numpy arrays for consistent handling
    data = np.asarray(data)
    centroids = np.asarray(centroidsInitial)
    
    # Validate inputs
    if len(centroids) != k:
        raise ValueError("Number of initial centroids must equal k.")
    if k <= 0:
        raise ValueError("k must be a positive integer.")
    
    n_iter = 0
    sse = float("inf")
    
    while n_iter < max_iterations:
        n_iter += 1
        
        # Calculate distances from each point to centroids
        distances = np.array([[distance(x, centroid) for centroid in centroids] for x in data])
        clustersIndex = np.argmin(distances, axis=1)
        
        # Update centroids
        centroidsNew = []
        for i in range(k):
            cluster_points = data[clustersIndex == i]
            if len(cluster_points) == 0:
                # Handle empty clusters by retaining the current centroid
                centroidsNew.append(centroids[i])
            else:
                centroidsNew.append(np.mean(cluster_points, axis=0))
        centroidsNew = np.array(centroidsNew)
        
        new_sse = SSE(data, centroidsNew, clustersIndex)
        
        # Check for convergence using tolerance
        if np.allclose(centroids, centroidsNew, atol=tol):
            break
        
        centroids = centroidsNew
        sse = new_sse
    
    end_time = time()
    
    return n_iter, sse, end_time - start_time


In [17]:
def kMeans_increaseSSE(data, k, initial_centroids, distance, max_iterations=100):
    start_time = time()
    
    # Input conversion and validation
    data = np.asarray(data)
    centroids = np.asarray(initial_centroids)
    
    if len(centroids) != k:
        raise ValueError("Initial centroids count must match k")
    if k <= 0 or not isinstance(k, int):
        raise ValueError("k must be positive integer")

    sse = float('inf')
    n_iter = 0
    
    while n_iter < max_iterations:
        n_iter += 1
        
        # Cluster assignment
        distance_matrix = np.array([[distance(x, c) for c in centroids] for x in data])
        clusters = np.argmin(distance_matrix, axis=1)
        
        # Centroid update with empty cluster handling
        updated_centroids = []
        for i in range(k):
            cluster_data = data[clusters == i]
            updated_centroids.append(
                centroids[i] if len(cluster_data) == 0 
                else cluster_data.mean(axis=0)
            )
        updated_centroids = np.array(updated_centroids)
        
        # SSE calculation
        new_sse = sum(
            distance(data[i], updated_centroids[clusters[i]])**2 
            for i in range(len(data))
        )
        
        # Termination check
        if new_sse >= sse:
            break
            
        centroids = updated_centroids
        sse = new_sse
    
    return (
        n_iter,
        sse,
        time() - start_time
    )


In [18]:
def kMeans_maxIterations(data, k, initial_centroids, distance, max_iterations=500):
    start_time = time()
    
    # Convert and validate inputs
    data = np.asarray(data)
    centroids = np.asarray(initial_centroids)
    
    if len(centroids) != k:
        raise ValueError("Initial centroids must match k value")
    if k <= 0 or not isinstance(k, int):
        raise ValueError("k must be positive integer")
    
    sse_history = []
    n_iter = 0
    
    while n_iter < max_iterations:
        n_iter += 1
        
        # Vectorized distance calculation
        dist_matrix = np.array([[distance(x, c) for c in centroids] for x in data])
        cluster_assignments = np.argmin(dist_matrix, axis=1)
        
        # Centroid update with stability checks
        new_centroids = np.zeros_like(centroids)
        for i in range(k):
            cluster_points = data[cluster_assignments == i]
            if cluster_points.size > 0:
                new_centroids[i] = cluster_points.mean(axis=0)
            else:
                new_centroids[i] = centroids[i]  # Maintain previous position
        
        # Calculate SSE using cluster assignments
        current_sse = sum(distance(pt, new_centroids[cluster])**2 
                         for pt, cluster in zip(data, cluster_assignments))
        sse_history.append(current_sse)
        
        centroids = new_centroids.copy()
    
    total_time = time() - start_time
    
    return (
        n_iter,  # Will always equal max_iterations
        sse_history[-1],
        total_time
    )


In [19]:
# Calculating all values for Euclidean Distance metric

euclidean_centroid_based_no_of_iterations, euclidean_centroid_based_sse, euclidean_centroid_based_time = kMeans_nochange(dataArray, k, centroidsInitial, euclideanDist)
euclidean_sse_based_no_of_iterations, euclidean_sse_based_sse, euclidean_sse_based_time = kMeans_increaseSSE(dataArray, k, centroidsInitial, euclideanDist)
euclidean_preset_based_no_of_iterations, euclidean_preset_based_sse, euclidean_preset_based_time = kMeans_maxIterations(dataArray, k, centroidsInitial, euclideanDist)

euclidean_df = pd.DataFrame({
    "Stop_Criteria": ["No Change in Centroid", "Increase in SSE", "Maximum preset value"],
    "No_of_Iterations": [euclidean_centroid_based_no_of_iterations, euclidean_sse_based_no_of_iterations, euclidean_preset_based_no_of_iterations],
    "Time_Taken": [euclidean_centroid_based_time, euclidean_sse_based_time, euclidean_preset_based_time],
    "SSE": [euclidean_centroid_based_sse, euclidean_sse_based_sse, euclidean_preset_based_sse]
})
euclidean_df[["Stop_Criteria", "No_of_Iterations", "Time_Taken"]].sort_values(by = ["No_of_Iterations", "Time_Taken"])

,Stop_Criteria,No_of_Iterations,Time_Taken
0,No Change in Centroid,45,45.799890
1,Increase in SSE,45,70.221050
2,Maximum preset value,500,540.948367


In [20]:
cosine_centroid_based_no_of_iterations, cosine_centroid_based_sse, cosine_centroid_based_time = kMeans_nochange(dataArray, k, centroidsInitial, cosineDist)
cosine_sse_based_no_of_iterations, cosine_sse_based_sse, cosine_sse_based_time = kMeans_increaseSSE(dataArray, k, centroidsInitial, cosineDist)
cosine_preset_based_no_of_iterations, cosine_preset_based_sse, cosine_preset_based_time = kMeans_maxIterations(dataArray, k, centroidsInitial, cosineDist)

cosine_df = pd.DataFrame({
    "Stop_Criteria": ["No Change in Centroid", "Increase in SSE", "Maximum preset value"],
    "No_of_Iterations": [cosine_centroid_based_no_of_iterations, cosine_sse_based_no_of_iterations, cosine_preset_based_no_of_iterations],
    "Time_Taken": [cosine_centroid_based_time, cosine_sse_based_time, cosine_preset_based_time],
    "SSE": [cosine_centroid_based_sse, cosine_sse_based_sse, cosine_preset_based_sse]
})
cosine_df[["Stop_Criteria", "No_of_Iterations", "Time_Taken"]].sort_values(by = ["No_of_Iterations", "Time_Taken"])

,Stop_Criteria,No_of_Iterations,Time_Taken
1,Increase in SSE,27,35.765459
0,No Change in Centroid,93,117.835032
2,Maximum preset value,500,629.787453


In [21]:
jaccard_centroid_based_no_of_iterations, jaccard_centroid_based_sse, jaccard_centroid_based_time = kMeans_nochange(dataArray, k, centroidsInitial, jaccardDist)
jaccard_sse_based_no_of_iterations, jaccard_sse_based_sse, jaccard_sse_based_time = kMeans_increaseSSE(dataArray, k, centroidsInitial, jaccardDist)
jaccard_preset_based_no_of_iterations, jaccard_preset_based_sse, jaccard_preset_based_time = kMeans_maxIterations(dataArray, k, centroidsInitial, jaccardDist)

jaccard_df = pd.DataFrame({
    "Stop_Criteria": ["No Change in Centroid", "Increase in SSE", "Maximum preset value"],
    "No_of_Iterations": [jaccard_centroid_based_no_of_iterations, jaccard_sse_based_no_of_iterations, jaccard_preset_based_no_of_iterations],
    "Time_Taken": [jaccard_centroid_based_time, jaccard_sse_based_time, jaccard_preset_based_time],
    "SSE": [jaccard_centroid_based_sse, jaccard_sse_based_sse, jaccard_preset_based_sse]
})
jaccard_df[["Stop_Criteria", "No_of_Iterations", "Time_Taken"]].sort_values(by = ["No_of_Iterations", "Time_Taken"])

,Stop_Criteria,No_of_Iterations,Time_Taken
1,Increase in SSE,20,42.799221
0,No Change in Centroid,98,186.898446
2,Maximum preset value,500,803.292277


**Q4: Compare the SSEs of Euclidean-K-means Cosine-K-means, Jarcard-K-means with respect to the following three terminating conditions: <br>
    • when there is no change in centroid position<br>
    • when the SSE value increases in the next iteration<br>
    • when the maximum preset value (e.g., 100) of iteration is complete**

In [22]:
euclidean_df[["Stop_Criteria", "SSE"]].sort_values(by = ["SSE"])

,Stop_Criteria,SSE
0,No Change in Centroid,2.546966e+10
1,Increase in SSE,2.546966e+10
2,Maximum preset value,2.547214e+10


In [23]:
cosine_df[["Stop_Criteria", "SSE"]].sort_values(by = ["SSE"])

,Stop_Criteria,SSE
1,Increase in SSE,6.962869e+02
2,Maximum preset value,6.968815e+02
0,No Change in Centroid,2.565005e+10


In [24]:
jaccard_df[["Stop_Criteria", "SSE"]].sort_values(by = ["SSE"])

,Stop_Criteria,SSE
2,Maximum preset value,3.653079e+03
1,Increase in SSE,3.661328e+03
0,No Change in Centroid,2.541436e+10
